# Decision Trees, Random Forests & Gradient Boosting — Foundations

**Goal:** compare the main tree-model families in one compact, DataCamp-style exercise.

Dataset: scikit-learn Breast Cancer Wisconsin diagnostic dataset. This is learning evidence, not a clinical decision system.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

RANDOM_STATE = 42
data = load_breast_cancer(as_frame=True)
X, y = data.data.copy(), data.target.copy()
print(X.shape)
display(X.head())
display(y.value_counts(normalize=True).rename('share'))


## 1. Shared holdout and baseline

All candidates use the same stratified train/test split so their holdout results are comparable.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
models = {
    'Dummy': DummyClassifier(strategy='most_frequent'),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
}
rows, predictions = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append({'model': name, 'accuracy': accuracy_score(y_test, pred), 'f1': f1_score(y_test, pred)})
comparison = pd.DataFrame(rows).sort_values('f1', ascending=False)
display(comparison.round(4))


## 2. Decision Tree — one interpretable recursive model

A single tree is easy to inspect but can overfit. `max_depth` is one direct control on complexity.


In [ ]:
tree = models['Decision Tree']
print('Depth:', tree.get_depth())
print('Leaves:', tree.get_n_leaves())
plt.figure(figsize=(18, 8))
plot_tree(
    tree, feature_names=X.columns, class_names=list(data.target_names),
    filled=False, max_depth=2, fontsize=8
)
plt.title('Top levels of the decision tree')
plt.show()


## 3. Random Forest — bagging and feature randomness

Random forests stabilise one tree by fitting many decorrelated trees. We tune forest size, depth, leaf size and feature sampling with cross-validation.


In [ ]:
rf_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    {
        'n_estimators': [200, 400],
        'max_depth': [None, 4, 8],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 0.7],
    },
    scoring='f1', cv=5, n_jobs=-1,
)
rf_search.fit(X_train, y_train)
rf_pred = rf_search.predict(X_test)
print(rf_search.best_params_)
print('Tuned RF accuracy:', accuracy_score(y_test, rf_pred))
print('Tuned RF F1:', f1_score(y_test, rf_pred))


In [ ]:
importance = (
    pd.Series(rf_search.best_estimator_.feature_importances_, index=X.columns)
    .sort_values(ascending=False).head(12)
)
display(importance.rename('importance').to_frame())
importance.sort_values().plot(kind='barh', figsize=(8, 5))
plt.title('Random-forest feature importance')
plt.xlabel('Importance')
plt.show()
ConfusionMatrixDisplay.from_predictions(y_test, rf_pred, display_labels=data.target_names)
plt.title('Tuned random forest — confusion matrix')
plt.show()


## 4. Gradient Boosting — sequential error correction

Boosting builds trees sequentially so later trees focus on errors left by earlier ones. The larger portfolio extends this family with **XGBoost** and **CatBoost**.


In [ ]:
gb = models['Gradient Boosting']
gb_importance = pd.Series(gb.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
display(gb_importance.rename('gradient_boosting_importance').to_frame())
example = X_test.iloc[[0]]
print('Example tuned-forest prediction:', data.target_names[int(rf_search.predict(example)[0])])


## Takeaway

- **Decision Tree:** one interpretable recursive partition; constrain complexity to reduce overfitting.
- **Random Forest:** bagging and feature randomness stabilise many trees.
- **Gradient Boosting:** sequential trees correct prior errors.
- **XGBoost / CatBoost:** advanced boosting approaches demonstrated in the professional portfolio.

Together with the SVM foundation, this closes the major classical-ML family coverage expected for junior/graduate roles.
